In [8]:
# import all libraries
import pandas as pd
import json
from sklearn.metrics import classification_report
from transformers import RobertaModel
from transformers import RobertaTokenizerFast
import torch, torchvision
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import Dataset
from torch import nn
from tqdm import tqdm

In [9]:
# load the data
with open("../01_data/annotations_reduced.json", "r") as f:
    data = json.load(f)

# initialize dictionary for the sentiment classes
sent_dict = set()

# loop through all sentences
for task in data:
    if task["annotations"]:
        for annotation in task["annotations"]:
            label = annotation["tag"][3:]
            sent_dict.add(label)

# sort the tag dictionary
label_list = sorted(sent_dict)

# dictionaries that convert from id to tag and vice versa
label_to_id = {tag: i for i, tag in enumerate(label_list)}
id_to_label = {id: label for label, id in label_to_id.items()}

In [10]:
# get all data with annotations
data_with_annotations = []
for task in data:
    if task["annotations"]:
        data_with_annotations.append(task)

# split into training and test dataset
split_idx = int(len(data_with_annotations) * 0.75)
train_dataset = data_with_annotations[:split_idx]
test_dataset = data_with_annotations[split_idx:]

class SpanDataset(Dataset):
    def __init__(self, data, tokenizer, label2id, max_len=128):

        self.dataset = []

        for task in data:
            # get the sentence and all annotations
            text = task["sentence"]
            spans = task["annotations"]

            # tokenize and get input ids and attention mask
            encoding = tokenizer(text, return_offsets_mapping=True, truncation=True, max_length=max_len)
            input_ids = encoding["input_ids"]
            attention_mask = encoding["attention_mask"]
            offsets = encoding["offset_mapping"]

            for span in spans:
                span_start = span["start"]
                span_end = span["end"]
                label = label2id[span["tag"][3:]]

                # find the token indices corresponding to the span
                span_token_indices = [i for i, (start, end) in enumerate(offsets) if start >= span_start and end <= span_end]

                # apply padding manually
                input_ids = input_ids + [1] * (max_len - len(input_ids))
                attention_mask = attention_mask + [0] * (max_len - len(attention_mask))

                self.dataset.append({
                    "input_ids": torch.tensor(input_ids, dtype=torch.long),
                    "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
                    "label": torch.tensor(label, dtype=torch.long),
                    "span_token_indices": torch.tensor(span_token_indices, dtype=torch.long)
                })
                
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        return self.dataset[idx]
    
def collate_fn(batch):
    input_ids = torch.stack([item["input_ids"] for item in batch])
    attention_masks = torch.stack([item["attention_mask"] for item in batch])
    labels = torch.stack([item["label"] for item in batch])
    span_token_indices = [item["span_token_indices"] for item in batch]

    return {
        "input_ids": input_ids,
        "attention_masks": attention_masks,
        "span_token_indices": span_token_indices,
        "labels": labels
    }

tokenizer = RobertaTokenizerFast.from_pretrained("roberta-base")
train_dataset_tensor = SpanDataset(data=train_dataset, tokenizer=tokenizer, label2id=label_to_id)
test_dataset_tensor = SpanDataset(data=test_dataset, tokenizer=tokenizer, label2id=label_to_id)
train_dataloader = DataLoader(train_dataset_tensor, batch_size=16, shuffle=True, collate_fn=collate_fn)
test_dataloader = DataLoader(test_dataset_tensor, batch_size=16, shuffle=True, collate_fn=collate_fn)

In [11]:
class SpanClassifier(nn.Module):
    def __init__(self, pretrained_model="roberta-base", num_labels=3):
        super().__init__()
        self.roberta = RobertaModel.from_pretrained(pretrained_model)
        self.classifier = nn.Linear(self.roberta.config.hidden_size, num_labels)
        self.dropout = nn.Dropout(0.1)
    
    def forward(self, input_ids, attention_mask, span_token_indices):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = outputs.last_hidden_state

        # pool span embeddings
        span_embeddings = []
        for i, indices in enumerate(span_token_indices):
            indices = indices.to(last_hidden.device)
            token_embeds = last_hidden[i, indices]  # select tokens for this span
            pooled = token_embeds.mean(dim=0)       # mean pooling
            span_embeddings.append(pooled)
        
        span_embeddings = torch.stack(span_embeddings)  # shape: (batch_size, hidden_size)
        span_embeddings = self.dropout(span_embeddings)
        logits = self.classifier(span_embeddings)      # shape: (batch_size, num_labels)
        return logits

In [14]:
# use mps if available, otherwise cpu
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = SpanClassifier(num_labels=3).to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

epochs = 10

model.train()

for epoch in range(epochs):

    # print the epoch number
    print(f"Epoch {epoch + 1}/{epochs}")

    # initialize training loss for the epoch
    total_loss = 0
    progress_bar = tqdm(train_dataloader, desc="Training")
    
    for batch in progress_bar:

        # extract all batch data and move to respective device
        input_ids = batch["input_ids"].to(device)
        attention_masks = batch["attention_masks"].to(device)
        span_token_indices = batch["span_token_indices"]
        labels = batch["labels"].to(device)

        # clear the old gradient
        model.zero_grad()

         # run data through the model and save the logits, use mps with mixed precision (16 bit floating point for forward pass)
        with torch.autocast(device_type="mps", dtype=torch.float16):
            logits = model(input_ids=input_ids, attention_mask=attention_masks, span_token_indices=span_token_indices)
            loss = criterion(logits, labels)
            total_loss += loss.item()

        # compute gradients by backpropagation and update all model weigths
        loss.backward()
        clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        progress_bar.set_postfix(loss=loss.item())

    # get the average training loss per batch and print
    avg_loss = total_loss / len(train_dataloader)
    print(f"Average training loss: {avg_loss:.4f}")

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/10


Training: 100%|██████████| 47/47 [00:27<00:00,  1.72it/s, loss=0.491]


Average training loss: 0.7717
Epoch 2/10


Training: 100%|██████████| 47/47 [00:19<00:00,  2.41it/s, loss=0.821]


...

Average training loss: 0.0361
Epoch 9/10


Training: 100%|██████████| 47/47 [00:20<00:00,  2.34it/s, loss=0.000658]


Average training loss: 0.0389
Epoch 10/10


Training: 100%|██████████| 47/47 [00:20<00:00,  2.31it/s, loss=5.59e-5] 

Average training loss: 0.0110


In [16]:
# set the model to evaluation mode (no loss calculation)
model.eval()

# initialize empty lists for true and predicted labels
true_classes = []
pred_classes = []

# proceed without calculating gradients
with torch.no_grad():

    # loop through all batches in the test dataloader
    for batch in test_dataloader:

        # move all batch data to respective device
        input_ids = batch["input_ids"].to(device)
        attention_masks = batch["attention_masks"].to(device)
        span_token_indices = batch["span_token_indices"]
        labels = batch["labels"].to(device)

        # get model outputs, get logits and get the class labels for the max logit
        logits = model(input_ids=input_ids,
                       attention_mask=attention_masks,
                       span_token_indices=span_token_indices)
        predictions = torch.argmax(logits, dim=1)

        # append true and predicted labels
        for i in range(len(labels)):
            true_classes.append(id_to_label[labels[i].item()])
            pred_classes.append(id_to_label[predictions[i].item()])

# print classification report
print(classification_report(true_classes, pred_classes))

              precision    recall  f1-score   support

         neg       0.92      1.00      0.96        11
     neutral       0.66      0.61      0.64        77
         pos       0.83      0.85      0.84       172

    accuracy                           0.79       260
   macro avg       0.80      0.82      0.81       260
weighted avg       0.78      0.79      0.79       260

